# 补充竞品航线价格

客流不仅仅取决于航班基本信息和自己的票价，还取决于竞争航班的票价

由于竞品全航线没有经济舱的单独数据，我们暂且用所有票的平均票价（经济舱和商务舱）来代替，效果可能没有使用经济舱票价好，但是这也要是眼下最好的处理方式了

竞争航班是指from和to相同的航线，起飞时间相差在4小时以内的价格最便宜的那趟航班作为竞争航班

如果4小时内没有from和to相同的航班，则以起飞时间最近的那趟航班作为竞争航班

如果历史上都没有from和to相同的航班，那么假定竞争航班票价和该航班票价相同




# 全市场票价计算

In [1]:
import pandas as pd
import numpy as np

# 读取CSV文件

df_all = pd.read_csv('../../data_hh/海航系销售结果数据_2023-01-01_2024-12-31.csv',index_col=0)
df_hh = pd.read_csv('../../data_hh/预处理的数据/pre_2023-2024.csv')


# 显示前两行数据以确保正确加载
print(df_all.shape)
print(df_all.head(5))
print(df_all.tail(5))

# 显示前两行数据以确保正确加载
print(df_hh.shape)
print(df_hh.head(5))
print(df_hh.tail(5))

(8798480, 16)
     flt_date segment flt_no      route    a    b    c bd_type  dep_time  cap  \
0  2023-01-01  TFULXA   111T  TFUGZGLXA  TFU  GZG  LXA      未知  18:00:00    0   
1  2023-01-01  GZGLXA   111T  TFUGZGLXA  TFU  GZG  LXA      未知  20:00:00    0   
2  2023-01-01  TFUGZG   111T  TFUGZGLXA  TFU  GZG  LXA      未知  18:00:00    0   
3  2023-01-01  LXAMIG   3017     LXAMIG  LXA  MIG  NaN      窄体  13:25:00  132   
4  2023-01-01  MIGLXA   3018     MIGLXA  MIG  LXA  NaN      窄体  16:15:00  132   

  aircraft  legs  leg_no  duration   tkt_rev  pax  
0      NaN     3       3      0.00       0.0    0  
1      NaN     3       2      3.00       0.0    0  
2      NaN     3       1      1.00       0.0    0  
3      319     1       1      1.70  223630.0  127  
4      319     1       1      2.28   76854.0   60  
           flt_date segment flt_no      route    a    b    c bd_type  \
8798475  2024-12-20  CGQKWE   8037  CGQTNAKWE  CGQ  TNA  KWE      窄体   
8798476  2024-12-20  CGQNNG   9344  CGQHFEN

## 部分字段统计情况

## 处理pax

In [2]:
# 统计 'pax' 字段中缺失值的行数
missing_pax = df_all[df_all['pax'].isna()]

# 统计 'pax' 字段中0值的行数
zero_pax = df_all[df_all['pax'] == 0]

# 输出统计结果
num_missing = missing_pax.shape[0]
num_zero = zero_pax.shape[0]

print(df_all.shape[0])
print(f"缺失值的行数: {num_missing}")
print(f"为0的行数: {num_zero}")

8798480
缺失值的行数: 0
为0的行数: 217917


In [3]:
# 删除 'pax' 字段为缺失值或为0的行
df_all = df_all.dropna(subset=['pax'])  # 删除pax列中的缺失值行
df_all = df_all[df_all['pax'] != 0]  # 删除pax列中为0的行

# 查看删除后的DataFrame行数
num_rows_after_cleanup = df_all.shape[0]
print(f"删除缺失值或为0的行后，DataFrame一共有 {num_rows_after_cleanup} 行")

删除缺失值或为0的行后，DataFrame一共有 8580563 行


## 处理tht_rev

In [4]:
# 统计 'unit_price' 列中为 0 的行数
zero_count = (df_all['tkt_rev'] == 0).sum()

# 统计 'unit_price' 列中为 NaN 的行数
nan_count = df_all['tkt_rev'].isna().sum()

# 输出结果
print(f"tkt_rev 列中为 0 的行数: {zero_count}")
print(f"tkt_rev 列中为 NaN 的行数: {nan_count}")

tkt_rev 列中为 0 的行数: 1438
tkt_rev 列中为 NaN 的行数: 0


In [5]:
# 过滤出 'tkt_rev' 为 0 的行
zero_tkt_rev = df_all[df_all['tkt_rev'] == 0]

# 统计这些 'tkt_rev' 为 0 的行中 'leg_no' 字段的不同取值及其数量
leg_no_counts = zero_tkt_rev['leg_no'].value_counts()

# 输出统计结果
print(f"tkt_rev 为 0 的行中，'leg_no' 字段的不同取值及其数量：")
print(leg_no_counts)

tkt_rev 为 0 的行中，'leg_no' 字段的不同取值及其数量：
leg_no
3    1121
1     225
2      91
6       1
Name: count, dtype: int64


In [6]:
df_all = df_all[df_all['tkt_rev'] != 0]

# 计算单价 'unit_price'，即 tkt_rev 除以 pax
df_all['unit_price'] = df_all['tkt_rev'] / df_all['pax']

# 删除 'tkt_rev' 列
df_all = df_all.drop(columns=['tkt_rev'])

# 查看结果
print(df_all.shape)
print(df_all.head(2))
print(df_all.tail(2))

(8579125, 16)
     flt_date segment flt_no   route    a    b    c bd_type  dep_time  cap  \
3  2023-01-01  LXAMIG   3017  LXAMIG  LXA  MIG  NaN      窄体  13:25:00  132   
4  2023-01-01  MIGLXA   3018  MIGLXA  MIG  LXA  NaN      窄体  16:15:00  132   

  aircraft  legs  leg_no  duration  pax   unit_price  
3      319     1       1      1.70  127  1760.866142  
4      319     1       1      2.28   60  1280.900000  
           flt_date segment flt_no   route    a    b    c bd_type  dep_time  \
8798478  2024-12-25  SHAPKX   6874  SHAPKX  SHA  PKX  NaN      窄体  09:40:00   
8798479  2024-12-29  TFUPVG   5296  TFUPVG  TFU  PVG  NaN      窄体  07:15:00   

         cap aircraft  legs  leg_no  duration  pax  unit_price  
8798478  175      321     1       1      1.85  152  624.013158  
8798479  158      320     1       1      2.02  139  412.719424  


# 为海航数据补充竞争航线价格

In [7]:
import pandas as pd
import numpy as np
from datetime import timedelta
from tqdm import tqdm  # 导入 tqdm

# 假设 df_all 和 df_hh 已经加载
# df_all = pd.read_csv('路径')  # 已加载的数据
# df_hh = pd.read_csv('路径')  # 已加载的数据

# 确保 df_all 和 df_hh 中 'dep_time' 和 'flt_date' 是正确的 datetime 类型
df_all['dep_time'] = pd.to_datetime(df_all['flt_date'].astype(str) + ' ' + df_all['dep_time'].astype(str), format='%Y-%m-%d %H:%M:%S')
df_hh['dep_time'] = pd.to_datetime(df_hh['flt_date'].astype(str) + ' ' + df_hh['dep_time'].astype(str), format='%Y-%m-%d %H:%M:%S')

# 1. 按 'segment' 字段分组
df_all_grouped = df_all.groupby('segment')

# 2. 准备一个空的列表用于存储竞争航班的票价
competitor_prices = []

# 3. 使用 tqdm 包装 df_hh.iterrows() 来显示进度条
for idx, row_hh in tqdm(df_hh.iterrows(), total=df_hh.shape[0], desc="Processing df_hh rows"):
    segment_hh, dep_time_hh = row_hh['segment'], row_hh['dep_time']
    
    # 获取与当前航班相同的 segment 的所有航班
    df_competing = df_all_grouped.get_group(segment_hh).copy() if segment_hh in df_all_grouped.groups else pd.DataFrame()

    if df_competing.empty:
        # 如果没有相同的航班，直接返回当前航班的票价
        competitor_prices.append(row_hh['unit_price'])
        continue
    
    # 4. 计算时间差
    time_diff = abs(df_competing['dep_time'] - dep_time_hh)
    
    # 5. 筛选出起飞时间与 df_hh 当前航班相差不超过 4 小时的航班
    df_competing.loc[:, 'time_diff'] = time_diff
    df_competing_4h = df_competing[df_competing['time_diff'].abs() <= timedelta(hours=4)]  # 使用 abs() 来计算绝对时间差
    
    if not df_competing_4h.empty:
        # 6. 如果有符合条件的航班，选择最便宜的
        cheapest_competing = df_competing_4h.loc[df_competing_4h['unit_price'].idxmin()]
        competitor_prices.append(cheapest_competing['unit_price'])
    else:
        # 7. 如果没有符合条件的航班，选择最接近的航班（按时间差最小）
        closest_competing = df_competing.loc[df_competing['time_diff'].idxmin()]
        competitor_prices.append(closest_competing['unit_price'])

# 将竞争航班的票价添加到 df_hh
df_hh['competitor_price'] = competitor_prices

# 查看结果
print(df_hh[['segment', 'dep_time', 'unit_price', 'competitor_price']].head())

Processing df_hh rows: 100%|███████| 1564130/1564130 [3:02:07<00:00, 143.14it/s]


  segment            dep_time   unit_price  competitor_price
0  AATURC 2023-01-01 14:35:00   470.474227        539.272727
1  ACFURC 2023-01-01 22:40:00   454.925373        578.797468
2  ACFXIY 2023-01-01 18:00:00  1177.818182       1177.818182
3  AKAHGH 2023-01-01 12:55:00   669.090909        669.090909
4  AKUCGO 2023-01-01 13:10:00  1794.783133       1859.812121


In [8]:
df_hh.to_csv('data_with_competitor_prices.csv')

In [9]:
df_hh.head(50)

,flt_date,segment,flt_no,dep_time,cap,aircraft,legs,leg_no,duration,pax,a,b,c,unit_price,competitor_price
0,2023-01-01,AATURC,7558,2023-01-01 14:35:00,110,195,1,1,1.30,97,AAT,URC,NaN,470.474227,539.272727
1,2023-01-01,ACFURC,7470,2023-01-01 22:40:00,110,195,1,1,1.25,67,ACF,URC,NaN,454.925373,578.797468
2,2023-01-01,ACFXIY,769R,2023-01-01 18:00:00,0,190,3,3,4.75,22,ACF,TLQ,XIY,1177.818182,1177.818182
3,2023-01-01,AKAHGH,5248,2023-01-01 12:55:00,162,320,1,1,1.83,55,AKA,HGH,NaN,669.090909,669.090909
4,2023-01-01,AKUCGO,6250,2023-01-01 13:10:00,167,320,1,1,3.98,166,AKU,CGO,NaN,1794.783133,1859.812121
5,2023-01-01,AKUURC,7514,2023-01-01 12:00:00,110,195,1,1,1.45,70,AKU,URC,NaN,610.571429,795.772727
6,2023-01-01,AQGKMG,9960,2023-01-01 14:45:00,167,32N,3,2,3.02,65,NGB,AQG,KMG,870.000000,870.000000
7,2023-01-01,AQGKWE,6490,2023-01-01 15:50:00,166,32C,3,2,2.35,24,TAO,AQG,KWE,773.333333,773.333333
8,2023-01-01,AQGXIY,7634,2023-01-01 15:15:00,111,195,3,2,2.03,31,XMN,AQG,XIY,483.870968,483.870968
9,2023-01-01,BARCGO,3518,2023-01-01 21:50:00,172,738,1,1,2.73,78,BAR,CGO,NaN,595.538462,595.538462


# 进一步处理海航数据

In [10]:
df = df_hh

## 拆分 flt_date 为 year, month, day, weekday

In [11]:
# 将 'flt_date' 列转换为 datetime 格式
df['flt_date'] = pd.to_datetime(df['flt_date'])

# 提取年、月、日和星期几
df['year'] = df['flt_date'].dt.year
df['month'] = df['flt_date'].dt.month
df['day'] = df['flt_date'].dt.day
df['weekday'] = df['flt_date'].dt.weekday  # 0 = Monday, 6 = Sunday

# 删除原始的 'flt_date' 字段
df = df.drop(columns=['flt_date'])

print(df.shape)
print(df.head(5))
print(df.tail(5))

(1564130, 18)
  segment flt_no            dep_time  cap aircraft  legs  leg_no  duration  \
0  AATURC   7558 2023-01-01 14:35:00  110      195     1       1      1.30   
1  ACFURC   7470 2023-01-01 22:40:00  110      195     1       1      1.25   
2  ACFXIY   769R 2023-01-01 18:00:00    0      190     3       3      4.75   
3  AKAHGH   5248 2023-01-01 12:55:00  162      320     1       1      1.83   
4  AKUCGO   6250 2023-01-01 13:10:00  167      320     1       1      3.98   

   pax    a    b    c   unit_price  competitor_price  year  month  day  \
0   97  AAT  URC  NaN   470.474227        539.272727  2023      1    1   
1   67  ACF  URC  NaN   454.925373        578.797468  2023      1    1   
2   22  ACF  TLQ  XIY  1177.818182       1177.818182  2023      1    1   
3   55  AKA  HGH  NaN   669.090909        669.090909  2023      1    1   
4  166  AKU  CGO  NaN  1794.783133       1859.812121  2023      1    1   

   weekday  
0        6  
1        6  
2        6  
3        6  
4      

## 拆分 dep_time 为 hour, minute, second

In [12]:
# 从 'dep_time' 字段提取小时、分钟和秒
df['hour'] = df['dep_time'].dt.hour
df['minute'] = df['dep_time'].dt.minute

# 删除原始的 'dep_time' 字段
df = df.drop(columns=['dep_time'])



# 查看结果
print(df.shape)
print(df.head(5))
print(df.tail(5))

(1564130, 19)
  segment flt_no  cap aircraft  legs  leg_no  duration  pax    a    b    c  \
0  AATURC   7558  110      195     1       1      1.30   97  AAT  URC  NaN   
1  ACFURC   7470  110      195     1       1      1.25   67  ACF  URC  NaN   
2  ACFXIY   769R    0      190     3       3      4.75   22  ACF  TLQ  XIY   
3  AKAHGH   5248  162      320     1       1      1.83   55  AKA  HGH  NaN   
4  AKUCGO   6250  167      320     1       1      3.98  166  AKU  CGO  NaN   

    unit_price  competitor_price  year  month  day  weekday  hour  minute  
0   470.474227        539.272727  2023      1    1        6    14      35  
1   454.925373        578.797468  2023      1    1        6    22      40  
2  1177.818182       1177.818182  2023      1    1        6    18       0  
3   669.090909        669.090909  2023      1    1        6    12      55  
4  1794.783133       1859.812121  2023      1    1        6    13      10  
        segment flt_no  cap aircraft  legs  leg_no  duration 

## 拆分 segment 为 from 和 to

In [13]:
# 提取 'segment' 列的前三个字符作为 'from' 列
df['from'] = df['segment'].str[:3]

# 提取 'segment' 列的后三个字符作为 'to' 列
df['to'] = df['segment'].str[-3:]

# 删除原始的 'segment' 字段
df = df.drop(columns=['segment'])

# 查看结果
print(df.shape)
print(df.head(2))
print(df.tail(2))

(1564130, 20)
  flt_no  cap aircraft  legs  leg_no  duration  pax    a    b    c  \
0   7558  110      195     1       1      1.30   97  AAT  URC  NaN   
1   7470  110      195     1       1      1.25   67  ACF  URC  NaN   

   unit_price  competitor_price  year  month  day  weekday  hour  minute from  \
0  470.474227        539.272727  2023      1    1        6    14      35  AAT   
1  454.925373        578.797468  2023      1    1        6    22      40  ACF   

    to  
0  URC  
1  URC  
        flt_no  cap aircraft  legs  leg_no  duration  pax    a    b    c  \
1564128   7830  161      738     3       1      1.63  131  ZUH  WUH  URC   
1564129   7522  161      738     1       1      2.50  140  ZUH  XIY  NaN   

         unit_price  competitor_price  year  month  day  weekday  hour  \
1564128  313.687023        311.449153  2024     12   31        1    16   
1564129  536.621429        397.525180  2024     12   31        1    17   

         minute from   to  
1564128      20  ZUH  WU

## 修改competitor_price为差值

In [14]:
df['competitor_price'] = df['unit_price'] - df['competitor_price']

In [15]:
print(df.shape)
print(df.head(2))
print(df.tail(2))

(1564130, 20)
  flt_no  cap aircraft  legs  leg_no  duration  pax    a    b    c  \
0   7558  110      195     1       1      1.30   97  AAT  URC  NaN   
1   7470  110      195     1       1      1.25   67  ACF  URC  NaN   

   unit_price  competitor_price  year  month  day  weekday  hour  minute from  \
0  470.474227        -68.798500  2023      1    1        6    14      35  AAT   
1  454.925373       -123.872095  2023      1    1        6    22      40  ACF   

    to  
0  URC  
1  URC  
        flt_no  cap aircraft  legs  leg_no  duration  pax    a    b    c  \
1564128   7830  161      738     3       1      1.63  131  ZUH  WUH  URC   
1564129   7522  161      738     1       1      2.50  140  ZUH  XIY  NaN   

         unit_price  competitor_price  year  month  day  weekday  hour  \
1564128  313.687023          2.237870  2024     12   31        1    16   
1564129  536.621429        139.096249  2024     12   31        1    17   

         minute from   to  
1564128      20  ZUH  WU

## 保存当前数据

In [16]:
df.to_csv("./pre_2023-2024_with_comp.csv", index=False, encoding="utf-8")

# 拆分训练集测试集

In [17]:
import pandas as pd

# 筛选出 year=2024 且 month 在 7 到 10 之间的数据（测试集 test）
df_test = df[(df['year'] == 2024) & (df['month'].between(7, 10))]

# 筛选出 其他数据（训练集 train）
df_train = df[~((df['year'] == 2024) & (df['month'].between(7, 10)))]

# 保存两个数据集
df_train.to_csv("pre_2023-2024_with_comp_train.csv", index=False, encoding="utf-8")
df_test.to_csv("pre_2023-2024_with_comp_test.csv", index=False, encoding="utf-8")

print("✅ 数据已成功拆分并保存：")
print("- 训练集 (其他数据) → pre_2023-2024_with_comp_train.csv")
print("- 测试集 (2024 年 7-10 月数据) → pre_2023-2024_with_comp_test.csv")

✅ 数据已成功拆分并保存：
- 训练集 (其他数据) → pre_2023-2024_with_comp_train.csv
- 测试集 (2024 年 7-10 月数据) → pre_2023-2024_with_comp_test.csv


In [18]:
# 获取行数
train_rows = df_train.shape[0]
test_rows = df_test.shape[0]

# 输出结果

print(f"   - 数据行数: {train_rows} 行")
print(f"   - 前5行数据:")
display(df_train.head())


print(f"   - 数据行数: {test_rows} 行")
print(f"   - 前5行数据:")
display(df_test.head())

   - 数据行数: 1281375 行
   - 前5行数据:


,flt_no,cap,aircraft,legs,leg_no,duration,pax,a,b,c,unit_price,competitor_price,year,month,day,weekday,hour,minute,from,to
0,7558,110,195,1,1,1.30,97,AAT,URC,NaN,470.474227,-68.798500,2023,1,1,6,14,35,AAT,URC
1,7470,110,195,1,1,1.25,67,ACF,URC,NaN,454.925373,-123.872095,2023,1,1,6,22,40,ACF,URC
2,769R,0,190,3,3,4.75,22,ACF,TLQ,XIY,1177.818182,0.000000,2023,1,1,6,18,0,ACF,XIY
3,5248,162,320,1,1,1.83,55,AKA,HGH,NaN,669.090909,0.000000,2023,1,1,6,12,55,AKA,HGH
4,6250,167,320,1,1,3.98,166,AKU,CGO,NaN,1794.783133,-65.028989,2023,1,1,6,13,10,AKU,CGO


   - 数据行数: 282755 行
   - 前5行数据:


,flt_no,cap,aircraft,legs,leg_no,duration,pax,a,b,c,unit_price,competitor_price,year,month,day,weekday,hour,minute,from,to
1158407,7558,94,190,1,1,1.33,45,AAT,URC,NaN,481.555556,-119.744444,2024,7,1,0,12,30,AAT,URC
1158408,5248,162,322,1,1,2.10,143,AKA,HGH,NaN,902.167832,0.000000,2024,7,1,0,10,30,AKA,HGH
1158409,3524,169,738,1,1,4.52,154,AKU,CGO,NaN,1540.357143,283.690476,2024,7,1,0,14,15,AKU,CGO
1158410,6352,167,320,1,1,4.28,167,AKU,CGO,NaN,1345.449102,88.782435,2024,7,1,0,21,30,AKU,CGO
1158411,7516,166,320,1,1,1.33,181,AKU,URC,NaN,593.480663,-54.049640,2024,7,1,0,23,15,AKU,URC


# 统计航段价格信息

In [19]:
# 查看结果
print(df.shape)
print(df.head(2))
print(df.tail(2))

(1564130, 20)
  flt_no  cap aircraft  legs  leg_no  duration  pax    a    b    c  \
0   7558  110      195     1       1      1.30   97  AAT  URC  NaN   
1   7470  110      195     1       1      1.25   67  ACF  URC  NaN   

   unit_price  competitor_price  year  month  day  weekday  hour  minute from  \
0  470.474227        -68.798500  2023      1    1        6    14      35  AAT   
1  454.925373       -123.872095  2023      1    1        6    22      40  ACF   

    to  
0  URC  
1  URC  
        flt_no  cap aircraft  legs  leg_no  duration  pax    a    b    c  \
1564128   7830  161      738     3       1      1.63  131  ZUH  WUH  URC   
1564129   7522  161      738     1       1      2.50  140  ZUH  XIY  NaN   

         unit_price  competitor_price  year  month  day  weekday  hour  \
1564128  313.687023          2.237870  2024     12   31        1    16   
1564129  536.621429        139.096249  2024     12   31        1    17   

         minute from   to  
1564128      20  ZUH  WU

In [20]:
# 计算每对城市间直航航班的价格统计信息
price_stats = df.groupby(['from', 'to']).agg({
    'unit_price': ['mean', 'std', 'count']
}).round(2)

# 重命名列
price_stats.columns = ['平均价格', '价格标准差', '航班数量']

# 对只有一趟航班的航线,将标准差设为平均价格的十分之一
single_flight_routes = price_stats['航班数量'] == 1
price_stats.loc[single_flight_routes, '价格标准差'] = price_stats.loc[single_flight_routes, '平均价格'] / 10

# 按平均价格降序排序
price_stats = price_stats.sort_values('平均价格', ascending=False)

# 显示结果
print("每对城市间直航航班的价格统计：")
print(price_stats)

# 可选：保存结果到CSV文件
price_stats.to_csv('./route_price_stats.csv')

# 输出一些基本统计信息
print("\n基本统计信息：")
print(f"航线总数：{len(price_stats)}")
print(f"最高平均票价航线：{price_stats['平均价格'].max():.2f}")
print(f"最低平均票价航线：{price_stats['平均价格'].min():.2f}")
print(f"最大标准差航线：{price_stats['价格标准差'].max():.2f}")

每对城市间直航航班的价格统计：
             平均价格    价格标准差  航班数量
from to                         
ENY  KRL  2900.00  290.000     1
DNH  PVG  2860.22  286.022     1
PVG  DNH  2860.00  286.000     1
URC  NGQ  2533.65   70.830   211
NGQ  URC  2529.94   63.180   226
...           ...      ...   ...
HUO  TGO    67.35   22.020    26
TGO  HUO    66.42   19.820    30
AVA  LLB    56.54    8.370     8
KWL  WUZ    52.47   33.560    12
LLB  AVA    52.26    2.920     8

[3257 rows x 3 columns]

基本统计信息：
航线总数：3257
最高平均票价航线：2900.00
最低平均票价航线：52.26
最大标准差航线：1135.80
